# Full Forecasting Workflow

This notebook demonstrates a **complete end-to-end forecasting pipeline** using forecastbox.
We walk through every stage of a professional macro forecasting exercise:

1. **Problem Definition** — define the target variable, horizon, and data
2. **Auto-Forecast** — generate individual model forecasts automatically
3. **Baseline Comparison** — benchmark against naive and simple methods
4. **Forecast Combination** — combine models to improve accuracy
5. **Formal Evaluation** — apply statistical tests (DM, MCS, Mincer-Zarnowitz)
6. **Scenario Analysis** — build conditional forecasts under alternative policy paths
7. **Final Report** — consolidate results into a dashboard with fan chart

**Target**: Brazilian inflation (monthly), 12-month-ahead forecast.

**Datasets used**: `macro_brazil.csv` (Phase 1), `us_macro_quarterly.csv` (Phase 5)

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# forecastbox modules
from forecastbox.auto import AutoARIMA, AutoETS, AutoSelect
from forecastbox.combination import SimpleCombiner, WeightedCombiner, OLSCombiner
from forecastbox.evaluation import diebold_mariano, model_confidence_set, mincer_zarnowitz
from forecastbox.scenarios import SimpleVAR, ConditionalForecast, ScenarioBuilder, MonteCarlo, FanChart
from forecastbox.metrics import mae, rmse, mape, mase
from forecastbox.cv import expanding_window_cv

# Helpers
sys.path.insert(0, "..")
from utils.helpers import load_all_datasets

warnings.filterwarnings("ignore")
np.random.seed(42)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("All forecastbox modules loaded successfully.")

## 1. Problem Definition

**Objective**: Forecast Brazilian monthly inflation 12 months ahead.

We start with exploratory data analysis — time series plot, autocorrelation function (ACF),
and seasonal decomposition — to understand the key features of the series before modelling.

In [ ]:
# Load Brazilian macro dataset
datasets = load_all_datasets()
df_brazil = datasets["macro_brazil"]
print(f"Shape: {df_brazil.shape}")
print(f"Date range: {df_brazil.index[0]} to {df_brazil.index[-1]}")
print(f"Columns: {list(df_brazil.columns)}")
df_brazil.head()

In [ ]:
# Target: inflation
inflation = df_brazil["inflation"]
h = 12  # forecast horizon

# Train/test split: hold out last 12 months for evaluation
train = inflation.iloc[:-h]
test = inflation.iloc[-h:]
print(f"Train: {len(train)} obs ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} obs ({test.index[0].date()} to {test.index[-1].date()})")

# EDA: time series plot, ACF, seasonal pattern
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import seasonal_decompose

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: time series
axes[0].plot(train.index, train.values, "steelblue", linewidth=1.2, label="Train")
axes[0].plot(test.index, test.values, "darkorange", linewidth=1.2, label="Test")
axes[0].axvline(test.index[0], color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Brazilian Inflation (Monthly)", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: ACF
plot_acf(train.dropna(), lags=36, ax=axes[1], alpha=0.05)
axes[1].set_title("Autocorrelation Function", fontsize=12)

# Plot 3: monthly box plot for seasonality
monthly = train.groupby(train.index.month)
axes[2].boxplot([monthly.get_group(m).values for m in range(1, 13)],
                labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
axes[2].set_title("Seasonal Pattern (by Month)", fontsize=12)
axes[2].grid(True, alpha=0.3)

fig.suptitle("Exploratory Data Analysis: Brazilian Inflation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Step 1: Auto-Forecast

We use forecastbox's automatic model selection to generate individual forecasts:

- **AutoARIMA** — searches over ARIMA(p,d,q)(P,D,Q)[m] orders using AICc
- **AutoETS** — selects the best exponential smoothing model (error, trend, seasonal)
- **AutoSelect** — runs both families with cross-validation and picks the best overall

In [ ]:
# AutoARIMA
auto_arima = AutoARIMA(seasonal=True, m=12, stepwise=True, ic="aicc")
arima_result = auto_arima.fit(train)
arima_fc = arima_result.forecast(h)
print(f"AutoARIMA selected: {arima_result}")

# AutoETS
auto_ets = AutoETS(seasonal_period=12, ic="aicc")
ets_result = auto_ets.fit(train)
ets_fc = ets_result.forecast(h)
print(f"AutoETS selected: {ets_result}")

# AutoSelect (cross-validation based)
auto_select = AutoSelect(
    families=["arima", "ets"],
    cv_type="expanding",
    cv_horizon=h,
    cv_step=3,
    metric="rmse",
)
select_result = auto_select.fit(train, m=12)
select_fc = select_result.forecast(h)
print(f"AutoSelect winner: {select_result}")

# Summary table
model_table = pd.DataFrame({
    "Model": ["AutoARIMA", "AutoETS", "AutoSelect"],
    "MAE": [
        mae(test.values, arima_fc.point),
        mae(test.values, ets_fc.point),
        mae(test.values, select_fc.point),
    ],
    "RMSE": [
        rmse(test.values, arima_fc.point),
        rmse(test.values, ets_fc.point),
        rmse(test.values, select_fc.point),
    ],
})
print("\n=== Auto-Forecast Results ===")
print(model_table.to_string(index=False))

## 3. Step 2: Baseline Comparison

No forecasting exercise is complete without baselines. We compare against:

- **Naive** — last observed value repeated forward
- **Seasonal Naive** — same month from the previous year
- **Simple Moving Average (SMA)** — rolling mean of the last 12 observations

We use expanding-window temporal cross-validation to get reliable out-of-sample metrics.

In [ ]:
# Baseline forecasts on test set
# Naive: repeat last value
naive_fc = np.full(h, train.iloc[-1])

# Seasonal naive: same month last year
snaive_fc = train.iloc[-12:].values

# SMA(12)
sma_fc = np.full(h, train.iloc[-12:].mean())

# Collect all forecasts for comparison
all_forecasts = {
    "AutoARIMA": arima_fc.point,
    "AutoETS": ets_fc.point,
    "AutoSelect": select_fc.point,
    "Naive": naive_fc,
    "Seasonal Naive": snaive_fc,
    "SMA(12)": sma_fc,
}

# Temporal CV for baselines
def naive_model(y):
    """Simple wrapper for naive forecast."""
    class _Naive:
        def __init__(self, y): self._last = y.iloc[-1]
        def forecast(self, h, **kw):
            from forecastbox.core import Forecast
            return Forecast(point=np.full(h, self._last), model_name="Naive", horizon=h)
    return _Naive(y)

def snaive_model(y):
    class _SNaive:
        def __init__(self, y): self._season = y.iloc[-12:].values
        def forecast(self, h, **kw):
            from forecastbox.core import Forecast
            fc = np.tile(self._season, (h // 12) + 1)[:h]
            return Forecast(point=fc, model_name="SNaive", horizon=h)
    return _SNaive(y)

cv_naive = expanding_window_cv(train, naive_model, initial_window=60, horizon=h, step=6)
cv_snaive = expanding_window_cv(train, snaive_model, initial_window=60, horizon=h, step=6)

# Metrics table: test set
actual = test.values
metrics_rows = []
for name, fc in all_forecasts.items():
    metrics_rows.append({
        "Model": name,
        "MAE": round(mae(actual, fc), 4),
        "RMSE": round(rmse(actual, fc), 4),
        "MAPE": round(mape(actual, fc), 4),
    })

metrics_df = pd.DataFrame(metrics_rows).sort_values("RMSE")
print("=== All Models: Test-Set Metrics ===")
print(metrics_df.to_string(index=False))

print(f"\nBaseline CV (Naive)   — mean RMSE: {cv_naive.errors['rmse'].mean():.4f}")
print(f"Baseline CV (SNaive)  — mean RMSE: {cv_snaive.errors['rmse'].mean():.4f}")

## 4. Step 3: Forecast Combination

Combining forecasts typically outperforms any single model (Timmermann 2006).
We apply three strategies:

- **Simple Average** — equal weights
- **Inverse MSE** — weight inversely proportional to past MSE
- **Granger-Ramanathan (OLS)** — regression-based weights (optimal under MSE loss)

In [ ]:
# Prepare training forecasts for combination fitting
# Use a validation window from the training set
val_size = 24
train_part = inflation.iloc[:-(h + val_size)]
val_part = inflation.iloc[-(h + val_size):-h]

# Generate in-sample forecasts on validation period
arima_val = AutoARIMA(seasonal=True, m=12, stepwise=True).fit(train_part).forecast(val_size)
ets_val = AutoETS(seasonal_period=12).fit(train_part).forecast(val_size)

fc_train_list = [arima_val.point, ets_val.point]
actual_val = val_part.values

# 1. Simple average
simple_comb = SimpleCombiner(method="mean")
simple_comb.fit(fc_train_list, actual_val)
combined_simple = simple_comb.combine([arima_fc, ets_fc])

# 2. Inverse MSE
weighted_comb = WeightedCombiner(method="inverse_mse")
weighted_comb.fit(fc_train_list, actual_val)
combined_weighted = weighted_comb.combine([arima_fc, ets_fc])

# 3. Granger-Ramanathan (OLS)
ols_comb = OLSCombiner(intercept=False, constrained=True)
ols_comb.fit(fc_train_list, actual_val)
combined_ols = ols_comb.combine([arima_fc, ets_fc])

# Weights summary
weights_df = pd.DataFrame({
    "Method": ["Simple Average", "Inverse MSE", "Granger-Ramanathan"],
    "w(ARIMA)": [
        0.5,
        round(weighted_comb.weights_[0], 4),
        round(ols_comb.weights_[0], 4),
    ],
    "w(ETS)": [
        0.5,
        round(weighted_comb.weights_[1], 4),
        round(ols_comb.weights_[1], 4),
    ],
    "RMSE": [
        round(rmse(actual, combined_simple.point), 4),
        round(rmse(actual, combined_weighted.point), 4),
        round(rmse(actual, combined_ols.point), 4),
    ],
})
print("=== Combination Weights & Metrics ===")
print(weights_df.to_string(index=False))

## 5. Step 4: Formal Evaluation

Statistical tests give us rigorous answers to key questions:

- **Diebold-Mariano (DM)** — is one forecast significantly better than another?
- **Model Confidence Set (MCS)** — which models belong to the "best" set?
- **Mincer-Zarnowitz (MZ)** — are the forecasts well-calibrated (unbiased, efficient)?

In [ ]:
# Diebold-Mariano pairwise matrix
selected_models = {
    "AutoARIMA": arima_fc.point,
    "AutoETS": ets_fc.point,
    "Combined(OLS)": combined_ols.point,
    "Naive": naive_fc,
}

model_names = list(selected_models.keys())
n_models = len(model_names)
dm_matrix = pd.DataFrame(np.nan, index=model_names, columns=model_names)

for i in range(n_models):
    for j in range(n_models):
        if i != j:
            result = diebold_mariano(
                actual, selected_models[model_names[i]],
                selected_models[model_names[j]], h=1, loss="mse"
            )
            dm_matrix.iloc[i, j] = round(result.pvalue, 4)

print("=== Diebold-Mariano p-value Matrix ===")
print("(Row model vs Column model — low p-value means row is significantly different)")
print(dm_matrix.to_string())

# Model Confidence Set
mcs_result = model_confidence_set(
    actual, selected_models, alpha=0.10, loss="mse",
    n_boot=5000, seed=42
)
print(f"\n=== Model Confidence Set (alpha=0.10) ===")
print(f"Included models: {mcs_result.included_models}")
print(f"Excluded models: {mcs_result.excluded_models}")
print(f"Elimination order: {mcs_result.elimination_order}")
print(f"p-values: {mcs_result.pvalues}")

# Mincer-Zarnowitz for each model
print("\n=== Mincer-Zarnowitz Calibration Tests ===")
for name, fc in selected_models.items():
    mz = mincer_zarnowitz(actual, fc)
    calibrated = "Yes" if mz.pvalue > 0.05 else "No"
    print(f"{name:20s}: alpha={mz.alpha:.4f}, beta={mz.beta:.4f}, "
          f"F={mz.f_statistic:.3f}, p={mz.pvalue:.4f}, R2={mz.r_squared:.4f} — Calibrated: {calibrated}")

## 6. Step 5: Scenario Analysis

We estimate a **VAR(2)** model with GDP growth, inflation, and interest rate to generate
conditional forecasts under two alternative monetary policy scenarios:

- **Hawkish**: interest rate rises by +200 bps over 12 months
- **Dovish**: interest rate falls by -100 bps over 12 months

In [ ]:
# VAR model with 3 macro variables
var_vars = ["gdp_growth", "inflation", "interest_rate"]
endog = df_brazil[var_vars].dropna().values
var_model = SimpleVAR(endog, p_order=2, var_names=var_vars)
steps = 12

print(f"VAR({var_model.p_order}) with {var_model.k_vars} variables, {endog.shape[0]} observations")

# Unconditional forecast (baseline)
cf = ConditionalForecast(var_model, method="analytic")
baseline = cf.forecast(steps=steps, conditions=None, n_draws=1000, seed=42)

# Hawkish scenario: interest rate rises +200bps gradually
last_rate = df_brazil["interest_rate"].iloc[-1]
hawkish_rates = [last_rate + (200 / 100) * (t + 1) / steps for t in range(steps)]

# Dovish scenario: interest rate falls -100bps gradually
dovish_rates = [last_rate - (100 / 100) * (t + 1) / steps for t in range(steps)]

# Build scenarios
builder = ScenarioBuilder(var_model)
builder.add_scenario("baseline",
                     {"interest_rate": baseline["interest_rate"].point.tolist()},
                     description="Unconditional baseline")
builder.add_scenario("hawkish",
                     {"interest_rate": hawkish_rates},
                     description="Tightening: +200bps over 12 months")
builder.add_scenario("dovish",
                     {"interest_rate": dovish_rates},
                     description="Easing: -100bps over 12 months")

scenario_results = builder.run(steps=steps, n_draws=1000, seed=42)

# Plot scenarios
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {"baseline": "blue", "hawkish": "red", "dovish": "green"}
horizons = np.arange(1, steps + 1)

for idx, var in enumerate(var_vars):
    ax = axes[idx]
    for scen_name in ["baseline", "hawkish", "dovish"]:
        fc = scenario_results.get(scen_name, var)
        ax.plot(horizons, fc.point, "-o", color=colors[scen_name],
                label=scen_name.capitalize(), linewidth=2, markersize=4)
        if fc.lower_80 is not None:
            ax.fill_between(horizons, fc.lower_80, fc.upper_80,
                            alpha=0.1, color=colors[scen_name])
    ax.set_title(var.replace("_", " ").title(), fontsize=12)
    ax.set_xlabel("Horizon (months)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Monetary Policy Scenarios: Hawkish (+200bps) vs Dovish (-100bps)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Step 6: Final Report

We consolidate all results into a **dashboard** that includes:
- Model ranking by RMSE
- Summary metrics table
- Fan chart of the best forecast with prediction intervals
- Scenario comparison overlay

In [ ]:
# === FINAL DASHBOARD ===
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# --- Panel 1: Model Ranking (bar chart) ---
ax = axes[0, 0]
all_models_final = {
    "AutoARIMA": arima_fc.point,
    "AutoETS": ets_fc.point,
    "AutoSelect": select_fc.point,
    "Naive": naive_fc,
    "SNaive": snaive_fc,
    "SMA(12)": sma_fc,
    "Comb(Mean)": combined_simple.point,
    "Comb(InvMSE)": combined_weighted.point,
    "Comb(GR)": combined_ols.point,
}
ranking = sorted(all_models_final.items(), key=lambda x: rmse(actual, x[1]))
names_ranked = [r[0] for r in ranking]
rmses_ranked = [round(rmse(actual, r[1]), 4) for r in ranking]
bar_colors = ["green" if r < np.median(rmses_ranked) else "salmon" for r in rmses_ranked]
ax.barh(names_ranked, rmses_ranked, color=bar_colors, edgecolor="white")
ax.set_xlabel("RMSE")
ax.set_title("Model Ranking by RMSE", fontsize=12, fontweight="bold")
ax.invert_yaxis()
for i, v in enumerate(rmses_ranked):
    ax.text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)

# --- Panel 2: Metrics Table ---
ax = axes[0, 1]
ax.axis("off")
table_data = []
for name, fc in ranking[:5]:  # top 5
    table_data.append([
        name,
        f"{mae(actual, fc):.4f}",
        f"{rmse(actual, fc):.4f}",
        f"{mape(actual, fc):.4f}",
    ])
table = ax.table(
    cellText=table_data,
    colLabels=["Model", "MAE", "RMSE", "MAPE"],
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 1.8)
ax.set_title("Top-5 Models: Summary Metrics", fontsize=12, fontweight="bold", pad=20)

# --- Panel 3: Fan Chart of Best Model ---
ax = axes[1, 0]
best_name, best_fc = ranking[0]
# Use Monte Carlo on VAR for fan chart
mc = MonteCarlo(var_model, n_paths=1000, seed=42, parametric=True)
mc.simulate(steps=steps)
fan = mc.fan_chart(variable="inflation")
fan.plot(ax=ax, title=f"Inflation Fan Chart (VAR Monte Carlo)",
         color="steelblue", history_periods=24)
ax.set_ylabel("Inflation")
ax.set_xlabel("Period")

# --- Panel 4: Scenario Comparison ---
ax = axes[1, 1]
for scen_name in ["baseline", "hawkish", "dovish"]:
    fc = scenario_results.get(scen_name, "inflation")
    ax.plot(horizons, fc.point, "-o", color=colors[scen_name],
            label=scen_name.capitalize(), linewidth=2, markersize=4)
    if fc.lower_80 is not None:
        ax.fill_between(horizons, fc.lower_80, fc.upper_80,
                        alpha=0.15, color=colors[scen_name])
ax.set_title("Inflation Under Policy Scenarios", fontsize=12, fontweight="bold")
ax.set_xlabel("Horizon (months)")
ax.set_ylabel("Inflation")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.suptitle("=== FORECASTING DASHBOARD: Brazilian Inflation ===",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# MCS summary
print("\n=== Model Confidence Set ===")
print(f"Best models (alpha=0.10): {mcs_result.included_models}")
print(f"\nConclusion: The MCS retains {len(mcs_result.included_models)} model(s) "
      f"as statistically indistinguishable from the best.")

## Exercise 1: Replicate this workflow for US GDP growth

Load the `us_macro_quarterly.csv` dataset and replicate the full pipeline above for
US GDP growth. Use quarterly frequency (m=4), 8-quarter-ahead horizon, and build
hawkish/dovish scenarios conditioning on the `fed_funds` rate.

In [ ]:
# TODO: Exercise 1
# Steps:
# 1. Load us_macro_quarterly.csv from datasets["us_macro_quarterly"]
# 2. Target: gdp_growth, horizon=8, m=4 (quarterly)
# 3. AutoARIMA(seasonal=True, m=4) and AutoETS(seasonal_period=4)
# 4. Baselines: naive, seasonal naive (lag 4), SMA(4)
# 5. Combine with SimpleCombiner, WeightedCombiner, OLSCombiner
# 6. DM test, MCS, Mincer-Zarnowitz
# 7. VAR scenario: hawkish (+150bps fed_funds) vs dovish (-75bps)
# 8. Final dashboard

## Exercise 2: Add nowcasting step to the workflow

Extend this pipeline by adding a nowcasting step **before** the forecasting stage.
Use `mixed_freq.csv` with a DFM nowcaster to estimate the current-quarter GDP,
then condition the VAR forecast on this nowcast estimate.

In [ ]:
# TODO: Exercise 2
# Steps:
# 1. Load mixed_freq.csv
# 2. DFMNowcaster(n_factors=1, factor_lags=2, frequency_map=...)
# 3. Nowcast current-quarter GDP
# 4. Use nowcast as initial condition for VAR forecast
# 5. Compare pipeline with vs without nowcasting step